In [ ]:
import pandas as pd
import os

# 통합 data 폴더 경로 설정 (상대 경로 기준)
# 경로가 다르다면 실제 위치에 맞게 수정해주세요!
data_path = "../../data/Mobile Reviews Sentiment.csv" 

df = pd.read_csv(data_path)

# 1. 데이터 상단 5줄 확인
print("--- Head ---")
display(df.head())

# 2. 데이터 크기 및 컬럼 타입, 결측치 확인
print("\n--- Info ---")
df.info()

--- Head ---


,review_id,customer_name,age,brand,model,price_usd,price_local,currency,exchange_rate_to_usd,rating,...,verified_purchase,battery_life_rating,camera_rating,performance_rating,design_rating,display_rating,review_length,word_count,helpful_votes,source
0,1,Aryan Maharaj,45,Realme,Realme 12 Pro,337.31,₹27996.73,INR,83.00,2,...,True,1,1,3,2,1,46,7,1,Amazon
1,2,Davi Miguel Sousa,18,Realme,Realme 12 Pro,307.78,R$1754.35,BRL,5.70,4,...,True,3,2,4,3,2,74,12,5,Flipkart
2,3,Pahal Balay,27,Google,Pixel 6,864.53,₹71755.99,INR,83.00,4,...,True,3,5,3,2,4,55,11,8,AliExpress
3,4,David Guzman,19,Xiaomi,Redmi Note 13,660.94,د.إ2425.65,AED,3.67,3,...,False,1,3,2,1,2,66,11,3,Amazon
4,5,Yago Leão,38,Motorola,Edge 50,792.13,R$4515.14,BRL,5.70,3,...,True,3,3,2,2,1,73,12,0,BestBuy



--- Info ---
<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 25 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   review_id             50000 non-null  int64  
 1   customer_name         50000 non-null  str    
 2   age                   50000 non-null  int64  
 3   brand                 50000 non-null  str    
 4   model                 50000 non-null  str    
 5   price_usd             50000 non-null  float64
 6   price_local           50000 non-null  str    
 7   currency              50000 non-null  str    
 8   exchange_rate_to_usd  50000 non-null  float64
 9   rating                50000 non-null  int64  
 10  review_text           50000 non-null  str    
 11  sentiment             50000 non-null  str    
 12  country               50000 non-null  str    
 13  language              50000 non-null  str    
 14  review_date           50000 non-null  str    
 15  verified_purchas

In [6]:
# 1. 상위 브랜드 및 모델 분포 확인
print("--- Top Brands ---")
print(df['brand'].value_counts().head(10))

# 2. 평점(rating)과 감성(sentiment) 분포 확인
print("\n--- Rating Distribution ---")
print(df['rating'].value_counts().sort_index())

print("\n--- Sentiment Distribution ---")
print(df['sentiment'].value_counts())

--- Top Brands ---
brand
Xiaomi      7241
Google      7234
Apple       7144
OnePlus     7136
Realme      7132
Motorola    7061
Samsung     7052
Name: count, dtype: int64

--- Rating Distribution ---
rating
1     6377
2     9803
3    12449
4    14029
5     7342
Name: count, dtype: int64

--- Sentiment Distribution ---
sentiment
Positive    27540
Neutral     12549
Negative     9911
Name: count, dtype: int64


In [8]:
import re
import pandas as pd

# 1. 텍스트 정제 함수 정의
def clean_text(text):
    if not isinstance(text, str):
        return ""
    # 소문자 변환
    text = text.lower()
    # 특수문자 및 숫자 제거 (알파벳과 공백만 남기기 - 필요에 따라 조정)
    # 영어 리뷰 데이터이므로 알파벳 외 기호나 이모지를 정제.
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    # 연속된 공백 제거
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# 2. 리뷰 텍스트 컬럼에 적용
df['cleaned_review'] = df['review_text'].apply(clean_text)

# 3. 결과 확인 (원본 vs 정제본)
print("--- Before & After Cleaning ---")
print(df[['review_text', 'cleaned_review']].head(6))

--- Before & After Cleaning ---
                                         review_text  \
0     Not worth the money spent. Wouldn’t recommend.   
1  Absolutely love this phone! The camera is next...   
2  Loving the clean UI and fast updates. Loving i...   
3  Build quality feels solid and durable. No regr...   
4  Not bad for daily use but could be optimized. ...   
5  Battery easily lasts a day with heavy use. No ...   

                                      cleaned_review  
0        not worth the money spent wouldnt recommend  
1  absolutely love this phone the camera is next ...  
2  loving the clean ui and fast updates loving it...  
3  build quality feels solid and durable no regre...  
4  not bad for daily use but could be optimized a...  
5  battery easily lasts a day with heavy use no r...  


In [12]:
from sklearn.feature_extraction.text import CountVectorizer

# 1. 간단한 단어 토큰화 및 빈도수 상위 키워드 추출 (CountVectorizer 활용)
# 불용어(english stop words)를 제거하여 의미 있는 단어만 추출합니다.
cv = CountVectorizer(stop_words='english', max_features=20)
dtm = cv.fit_transform(df['cleaned_review'])

# 상위 20개 키워드와 빈도수 확인
sum_words = dtm.sum(axis=0)
word_freq = [(word, sum_words[0, idx]) for word, idx in cv.vocabulary_.items()]
word_freq = sorted(word_freq, key=lambda x: x[1], reverse=True)

print("--- Top 20 Keywords in Reviews ---")
for word, freq in word_freq:
    print(f"{word}: {freq}")

--- Top 20 Keywords in Reviews ---
use: 11148
worth: 10640
loving: 9754
absolutely: 9556
best: 8103
okay: 7946
buying: 7824
far: 7004
purchase: 6869
year: 6869
regrets: 6839
fast: 6400
quality: 5955
phone: 5953
smooth: 5539
feels: 5507
overall: 5482
better: 5405
average: 5382
fine: 5326


In [9]:
# 브랜드별 평균 평점 및 리뷰 개수 집계
brand_summary = df.groupby('brand').agg(
    review_count=('rating', 'count'),
    avg_rating=('rating', 'mean')
).reset_index().sort_values(by='review_count', ascending=False)

print("--- Brand Summary (Review Count & Avg Rating) ---")
print(brand_summary)

--- Brand Summary (Review Count & Avg Rating) ---
      brand  review_count  avg_rating
6    Xiaomi          7241    3.113382
1    Google          7234    3.122477
0     Apple          7144    3.129759
3   OnePlus          7136    3.129344
4    Realme          7132    3.138811
2  Motorola          7061    3.114148
5   Samsung          7052    3.113868
